# Lab | Transformers

---

**Note: for GPU acceleration, run this notebook in [Google Colab](https://colab.research.google.com/).**



### Section structure

1. The open-source ecosystem: increasing accessibility to machine learning (ML) software and hardware
2. Some simple code demonstrations
3. Q&A

## 1. Ease-of-use: Using Transformers in 3 lines of code


**Overview of different tasks that can be automated with ML**
* Key ingredients: (1) a model trained on a specific task; (2) input data (e.g. texts or images); (3) output produced by the model.
* Transformers are currently the most popular type of deep learning algorithm. Most tasks below are solved with Transformers. There might be other types of algorithms coming up in the medium term.



**Install the Transformers library & dependencies**

In [1]:
!pip install transformers  # The Transformers library from Hugging Face
!pip install sentencepiece
!pip install wikipedia
!pip install accelerate
!pip install tf-keras
!pip install torch

# NOTE: you might need to restart you jupyter kernel after installing the libraries

**The Hugging Face Pipeline**
* Makes automation of many NLP tasks possible in 3 lines of code
* Detailed documentation is available [here](https://huggingface.co/transformers/main_classes/pipelines.html)

In [2]:
# STEP 2 — Import the tools for NLP inference
from transformers import pipeline
import pandas as pd
import numpy as np
from pprint import pprint

# STEP 3 — Confirm the imports succeeded
print("Libraries imported successfully!")

/opt/anaconda3/envs/dft/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1029 14:54:21.025000 25543 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Libraries imported successfully!


Note : You might need more libraries or updates to run the cells below, if that is the case, follow the error messages and pip install accordingly. Chat gpt can help you if given the error messages.

### 2.1 Many models tailored to specific tasks


#### 2.1.1 Text classification

Let's select a popular text classification model in the [HF model hub](https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads).

Here we chose "cardiffnlp/twitter-roberta-base-irony".

We will classify text into ironic or non ironic.

In [4]:
# STEP 1 — Load a pretrained model for irony classification
pipeline_classification = pipeline(
    "text-classification",
    model="cardiffnlp/twitter-roberta-base-irony"
)

# STEP 2 — Define the text to analyse
text = "Well that workshop was totally worth my time..."

# STEP 3 — Ask the model to classify the text
output = pipeline_classification(text, top_k=2)

# STEP 4 — Display the classification scores
pd.DataFrame(output)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 30649.86it/s]


,label,score
0,irony,0.942439
1,non_irony,0.057561


Now that we have the model we can pass it a string and have it give us a classification.

Feel free to experiment with different sentences by changing the contents of the variable text

In [5]:
# STEP 1 — Define sentences with different possible meanings
sentences = [
    "I really enjoyed the workshop.",
    "Fantastic! Another three-hour meeting that could have been an email.",
    "The experiment was completed successfully.",
    "Wonderful, my computer crashed again!"
]

# STEP 2 — Classify each sentence
results = []

for sentence in sentences:
    prediction = pipeline_classification(sentence, top_k=1)[0]

    results.append({
        "Sentence": sentence,
        "Prediction": prediction["label"],
        "Score": prediction["score"]
    })

# STEP 3 — Present results in a table
pd.DataFrame(results)

,Sentence,Prediction,Score
0,I really enjoyed the workshop.,irony,0.581496
1,Fantastic! Another three-hour meeting that cou...,irony,0.914394
2,The experiment was completed successfully.,irony,0.862506
3,"Wonderful, my computer crashed again!",irony,0.979710


Let's make the output a little cleaner

As you can see, in a few lines of code and by leveraging an existing model we can classify text as ironic or non ironic. Now you have one more tool in your machine learning toolbox.

Remember that : 'when you only have a hammer everything is a nail'. But if we want to build a house (perform machine learning the right way), we need to use the right tool for the right job.

#### 2.1.2 Machine Translation

* Open source machine translation (MT) models enable you to translate between many different languages without Google Translate.
* [University of Helsinki](https://huggingface.co/Helsinki-NLP) uploaded models for more than 1000 language pairs to the Hugging Face hub
* [Facebook AI](https://huggingface.co/models?search=facebook+m2m) open-sourced several multi-lingual models
* The [EasyNMT library](https://github.com/UKPLab/EasyNMT), provides an easy wrapper for all these models
* Most machine translation models translate between two languages in one direction (e.g. German to English, but not English to German), some can translate in multiple directions.


In [8]:
# translation pipeline docs: https://huggingface.co/transformers/main_classes/pipelines.html#transformers.TranslationPipeline

# STEP 1 — Import the translation model and tokenizer
from transformers import (
    M2M100ForConditionalGeneration,
    M2M100Tokenizer
)
import torch

# STEP 2 — Identify the pretrained model
model_name = "facebook/m2m100_418M"

# STEP 3 — Load the tokenizer
# The tokenizer converts words into numerical tokens
tokenizer = M2M100Tokenizer.from_pretrained(model_name)

# STEP 4 — Load the pretrained translation model
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# STEP 5 — Set the model to evaluation mode
model.eval()

print("Translation model loaded successfully!")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 57632.00it/s]


Translation model loaded successfully!


M2M100 is a multilingual encoder-decoder (seq-to-seq) model trained for Many-to-Many multilingual translation.

The model that can directly translate between the 9,900 directions of 100 languages.

Here we specify to translate from German 'de' to English 'en'

In [9]:
# STEP 1 — Define the input sentence
german_text = "Ich lerne maschinelles Lernen und künstliche Intelligenz."

# STEP 2 — Specify the source language
tokenizer.src_lang = "de"

# STEP 3 — Convert the German sentence into tokens
inputs = tokenizer(german_text, return_tensors="pt")

# STEP 4 — Generate an English translation
with torch.no_grad():
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id("en")
    )

# STEP 5 — Decode tokens back into readable text
english_text = tokenizer.batch_decode(
    generated_tokens,
    skip_special_tokens=True
)[0]

# STEP 6 — Display the result
print("German:", german_text)
print("English:", english_text)

German: Ich lerne maschinelles Lernen und künstliche Intelligenz.
English: I learn machine learning and artificial intelligence.


Let's do the same but with and entire wikipedia page in german.

#### 2.1.3 Text Summarization

In [11]:
# STEP 1 — Import the model and tokenizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# STEP 2 — Select a pretrained summarisation model
model_name = "sshleifer/distilbart-cnn-12-6"

# STEP 3 — Load tokenizer and model
summary_tokenizer = AutoTokenizer.from_pretrained(model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# STEP 4 — Set evaluation mode
summary_model.eval()

print("Summarisation model loaded successfully!")

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100%|██████████| 358/358 [00:00<00:00, 63803.89it/s]


Summarisation model loaded successfully!


In [13]:
# STEP 1 — Tokenise the original passage
inputs = summary_tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=1024
)

# STEP 2 — Generate a summary with explicit length settings
with torch.no_grad():
    summary_ids = summary_model.generate(
        **inputs,
        max_length=70,
        min_length=20,
        do_sample=False
    )

# STEP 3 — Decode the summary
summary = summary_tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

# STEP 4 — Display the complete output
print("ORIGINAL TEXT:")
print(text.strip())

print("\nGENERATED SUMMARY:")
print(summary)

ORIGINAL TEXT:
Electrochemical carbon dioxide reduction is an emerging technology
that converts carbon dioxide into valuable chemicals and fuels
using electrical energy. The process can produce carbon monoxide,
formic acid, methane, ethylene, and ethanol depending on the
catalyst and reaction conditions. Copper-based catalysts are
particularly interesting because they can produce multicarbon
products. However, challenges remain regarding energy efficiency,
product selectivity, and long-term operational stability.

GENERATED SUMMARY:
 Carbon dioxide reduction is an emerging technology that converts carbon dioxide into valuable chemicals and fuels . The process can produce carbon monoxide, formic acid, methane, ethylene, and ethanol depending on the reaction conditions .


In [ ]:
import wikipedia

# STEP 1 — Identify our script to Wikipedia
# Replace the email with your own contact address
wikipedia.set_user_agent(
    "IronhackNLPStudy/1.0 (bnjaato@gmail.com)"
)

# STEP 2 — Select German Wikipedia
wikipedia.set_lang("de")

# STEP 3 — Retrieve an article summary
text = wikipedia.summary(
    "Donald Trump",
    auto_suggest=False
).replace("\n", " ")[:1000]

# STEP 4 — Display the retrieved text
print("Original German text:\n")
print(text)

Original German text:

Donald John Trump [ˈdɑn.əld dʒɑn tɹɐmp] (* 14. Juni 1946 in Queens, New York City) ist ein US-amerikanischer Politiker der Republikanischen Partei. Er war von 2017 bis 2021 der 45. und ist seit dem 20. Januar 2025 der 47. Präsident der Vereinigten Staaten. Außerdem ist er Unternehmer und ehemaliger Showmaster. Er gilt als einer der umstrittensten Politiker der US-Geschichte und ist der erste Präsident, gegen den zwei Amtsenthebungsverfahren angestrengt wurden und der sich nach seiner ersten Präsidentschaft mit mehreren Strafprozessen konfrontiert sieht – unter anderem wegen Verschwörung gegen die USA. Er ist der erste strafrechtlich verurteilte US-Präsident. Trump leitete von 1971 bis 2017 den Mischkonzern The Trump Organization. Trotz eines geerbten Vermögens und mehrerer Insolvenzen in der Immobilienbranche wurde er zum Sinnbild eines erfolgreichen Geschäftsmannes. Ab 2004 wurde er als Gastgeber von The Apprentice populär, einer bis 2015 ausgestrahlten Reality-

In [15]:
# STEP 1 — Set German as the source language
tokenizer.src_lang = "de"

# STEP 2 — Convert Wikipedia text into numerical tokens
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=512
)

# STEP 3 — Generate English tokens
with torch.no_grad():
    translated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id("en"),
        max_new_tokens=300
    )

# STEP 4 — Decode the English translation
text_translated = tokenizer.batch_decode(
    translated_tokens,
    skip_special_tokens=True
)[0]

# STEP 5 — Display the result
print("Original German text:\n", text)
print("\nTranslated English text:\n", text_translated)

[transformers] Both `max_new_tokens` (=300) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original German text:
 Donald John Trump [ˈdɑn.əld dʒɑn tɹɐmp] (* 14. Juni 1946 in Queens, New York City) ist ein US-amerikanischer Politiker der Republikanischen Partei. Er war von 2017 bis 2021 der 45. und ist seit dem 20. Januar 2025 der 47. Präsident der Vereinigten Staaten. Außerdem ist er Unternehmer und ehemaliger Showmaster. Er gilt als einer der umstrittensten Politiker der US-Geschichte und ist der erste Präsident, gegen den zwei Amtsenthebungsverfahren angestrengt wurden und der sich nach seiner ersten Präsidentschaft mit mehreren Strafprozessen konfrontiert sieht – unter anderem wegen Verschwörung gegen die USA. Er ist der erste strafrechtlich verurteilte US-Präsident. Trump leitete von 1971 bis 2017 den Mischkonzern The Trump Organization. Trotz eines geerbten Vermögens und mehrerer Insolvenzen in der Immobilienbranche wurde er zum Sinnbild eines erfolgreichen Geschäftsmannes. Ab 2004 wurde er als Gastgeber von The Apprentice populär, einer bis 2015 ausgestrahlten Reality-

#### 2.1.4 Named Entity Recognition

NER is a task that involves identifying and classifying specific entities in text into predefined categories, such as names of people, organizations, locations, dates, and more. 

For example, in the sentence "Apple Inc. was founded by Steve Jobs in California," NER would recognize "Apple Inc." as an organization, "Steve Jobs" as a person, and "California" as a location.

In [18]:
# docs for summarisation pipeline: https://huggingface.co/transformers/main_classes/pipelines.html#summarizationpipeline
# STEP 1 — Import the Hugging Face pipeline
from transformers import pipeline

# STEP 2 — Load the pretrained NER model
ner_model = pipeline(
    "token-classification",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

print("NER model loaded successfully!")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10293.98it/s]
[transformers] BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER model loaded successfully!


In [19]:
# STEP 1 — Define a scientific research sentence
scientific_text = (
    "Researchers at the University of Cambridge and "
    "the University of Duisburg-Essen are studying "
    "electrochemical energy conversion in Germany."
)

# STEP 2 — Extract named entities
scientific_entities = ner_model(scientific_text)

# STEP 3 — Display the results
pd.DataFrame(scientific_entities)

,entity_group,score,word,start,end
0,ORG,0.994779,University of Cambridge,19,42
1,ORG,0.985032,University of Duisburg,51,73
2,LOC,0.861520,Essen,74,79
3,LOC,0.999408,Germany,130,137


### 2.2. Universal models

The models mentioned above are designed to excel at a single specific task on a particular dataset. The key advantage of these models is their high performance and accuracy on that specific task and dataset. 

However, in real-world applications, the problems you'll face often require solving slightly different tasks, possibly with varied category definitions or applied to different types of texts.

Universal models can help address this challenge. Although they also focus on one task, the task is general or universal enough that many other tasks can be reformulated into it. Two examples of universal tasks are:

- Natural Language Inference (NLI): A task that can effectively solve a wide range of classification tasks by determining whether a given premise supports, contradicts, or is neutral with respect to a hypothesis.

- Token Generation: An even more universal task that can be applied to solve virtually any text-related task, including translation, summarization, and text completion.

These universal tasks enable the models to be versatile and adaptable to various problems beyond the specific ones they were initially trained on.

#### Zero-shot classification


Zero-shot classification is a technique where a model can categorize data into classes it has never seen before. 

Instead of relying on labeled examples for each class, the model understands the relationship between the input and the class descriptions, allowing it to make accurate predictions without needing specific training on those classes.

In [24]:
# STEP 1 — Import the pipeline
from transformers import pipeline

# STEP 2 — Load a pretrained zero-shot model
zero_shot_model = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

print("Zero-shot classification model loaded!")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7665.06it/s]


Zero-shot classification model loaded!


Here we will give the model a list of classes ('payment issues', 'travel advice', 'bug report') for it to classify our string.

In [25]:
# STEP 1 — Define the text
research_text = (
    "Copper-based catalysts enable the electrochemical "
    "conversion of carbon dioxide into multicarbon products."
)

# STEP 2 — Define possible categories
categories = [
    "Electrocatalysis",
    "Battery research",
    "Machine learning",
    "Organic synthesis"
]

# STEP 3 — Classify the text
result = zero_shot_model(
    research_text,
    candidate_labels=categories
)

# STEP 4 — Display the classification results
import pandas as pd

pd.DataFrame({
    "Category": result["labels"],
    "Score": result["scores"]
})

,Category,Score
0,Electrocatalysis,0.941666
1,Machine learning,0.025647
2,Organic synthesis,0.017006
3,Battery research,0.015681


## Exercise

Now it is your turn to go to the hugging face library https://huggingface.co/models?pipeline_tag=text-classification&sort=downloads

(you can select on the left menu of the website the type of NLP tasks you want models to perform)

- Find an NLP model that we have not used previously.

- Get some data from wikipedia or elsewhere.

- Perform inference with the model and print the result!

- Comment your code along the way, describe what your model does and what your end goal is from input to output.

Have fun!

In [26]:
# STEP 1 — Import the required libraries
from transformers import pipeline
import pandas as pd
import wikipedia

# STEP 2 — Choose a pretrained language detection model
model_name = "papluca/xlm-roberta-base-language-detection"

# STEP 3 — Load the model for text classification
language_detector = pipeline(
    "text-classification",
    model=model_name
)

print("Language detection model loaded successfully!")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12008.65it/s]


Language detection model loaded successfully!


In [27]:
# STEP 1 — Set an identifiable User-Agent
# Replace the example address with your contact email
wikipedia.set_user_agent(
    "IronhackLanguageLab/1.0 (bnjaato@gmail.com)"
)

# STEP 2 — Retrieve articles in different languages
samples = []

for language in ["de", "en", "fr"]:
    # Specify the Wikipedia language
    wikipedia.set_lang(language)

    # Retrieve text about the same subject
    article = wikipedia.summary(
        "Albert Einstein",
        sentences=3,
        auto_suggest=False
    )

    # Store the text and its known language
    samples.append({
        "Expected Language": language,
        "Text": article.replace("\n", " ")
    })

print("Wikipedia texts retrieved successfully!")
pd.DataFrame(samples)

Wikipedia texts retrieved successfully!


,Expected Language,Text
0,de,Albert Einstein (* 14. März 1879 in Ulm; † 18....
1,en,Albert Einstein (14 March 1879 – 18 April 1955...
2,fr,Albert Einstein (prononcé en allemand [ˈalbɐt ...


In [28]:
# STEP 1 — Create a list to store predictions
results = []

# STEP 2 — Classify each Wikipedia text
for sample in samples:

    # Ask the pretrained model to detect the language
    prediction = language_detector(
        sample["Text"],
        truncation=True
    )[0]

    # Save the prediction and confidence score
    results.append({
        "Expected Language": sample["Expected Language"],
        "Predicted Language": prediction["label"],
        "Score": prediction["score"],
        "Correct": (
            prediction["label"] == sample["Expected Language"]
        )
    })

# STEP 3 — Display the prediction table
results_df = pd.DataFrame(results)

results_df

,Expected Language,Predicted Language,Score,Correct
0,de,de,0.994187,True
1,en,en,0.930391,True
2,fr,fr,0.994106,True


In [29]:
# STEP 1 — Calculate the fraction of correct predictions
accuracy = results_df["Correct"].mean()

# STEP 2 — Display the evaluation result
print(f"Accuracy on our sample: {accuracy:.2%}")

# STEP 3 — Show any incorrect predictions
print("\nIncorrect predictions:")
display(results_df[~results_df["Correct"]])

Accuracy on our sample: 100.00%

Incorrect predictions:


,Expected Language,Predicted Language,Score,Correct


### Exercise: Automatic Language Detection Using XLM-RoBERTa

**Objective:** The aim of this exercise is to apply a pretrained Transformer model to automatically identify the language of text collected from Wikipedia.

**Selected Model:** `papluca/xlm-roberta-base-language-detection`

**Methodology:** Text samples were retrieved from German, English and French Wikipedia. The XLM-RoBERTa model was loaded using the Hugging Face `text-classification` pipeline and applied to each sample without additional training.

**Input:** Wikipedia text in different languages.

**Output:** The predicted language code and the model's classification score.

**Evaluation:** The predicted languages were compared against the known languages of the Wikipedia articles.

**Conclusion:** This exercise demonstrates how a pretrained multilingual Transformer can be used for language identification. Its predictions should be evaluated against known labels, and high model scores do not guarantee correctness.